# Mapping Stations to Substations

Nearest-Neighbor matching as a geographic proxy for associating stations with their closest substations.

**Assessment**

Use nearest-neighbor matching as a *geographic proxy*, not evidence of electrical supply. The available datasets contain coordinates, but no feeder, transformer, service-territory, or utility connectivity identifier. This is consistent with the distinction already made in `evaluation.tex`: the physically supplying substation can differ from the closest one.

For your stated assumption, clarify that it means:

- Each EV station maps to exactly one substation.
- A substation may map to many EV stations.

A literal one-to-one/bijective match would be artificial and likely infeasible when dataset counts differ.

**Recommended criterion**

1. Reproject both GeoDataFrames from WGS84 to `EPSG:2263`, as `03-substations.ipynb` already uses it for ontology coordinates. Distances are then in US survey feet.
2. For every station, select the geographically nearest Con Edison substation: this is effectively `KNN` with $k=1$.
3. Calculate the nearest and second-nearest distances. Retain:
   - `substation_id`
   - `distance_ft`
   - `second_nearest_distance_ft`
   - `distance_margin_ft = d_2 - d_1`
   - `distance_ratio = d_1 / d_2`
   - `mapping_method = "nearest_substation_geographic_proxy"`
4. Resolve exact-distance ties deterministically by OSM `id`.
5. Do not silently discard distant stations under the forced-assignment assumption. Instead, flag them as `review_required` based on a predeclared maximum distance and/or a small nearest-vs-second-nearest margin.

Use $k=2$ for quality assessment, but select the $k=1$ result. A full KNN/ML implementation is unnecessary; GeoPandas `sjoin_nearest` is clearer and appropriate for this spatial join.

```python
stations_2263 = ev_stations.to_crs(epsg=2263)
substations_2263 = substations.to_crs(epsg=2263)

mapping = stations_2263.sjoin_nearest(
    substations_2263[["substation_id", "name", "geometry"]],
    how="left",
    distance_col="distance_ft",
)
```

**Important caveats**

- EV coordinates are geocoded street addresses, while OSM substations may be representative points of polygonal sites. Small distance differences should not be overinterpreted.
- Do not filter candidates by borough: electrical service may cross borough boundaries, and doing so imposes an unverified constraint.
- Substation `voltage` and `substation` type are useful descriptive fields, but they cannot establish compatibility with a station because station load voltage is absent.
- The mapping should be named `nearest_substation` or `proximity_inferred_substation`, never `connected_substation`.

For the ontology, retain this as an explicit inferred relationship/table with method and distance provenance. A later CIM graph relationship, when feeder/topology data becomes available, should supersede this geographic baseline.

In [1]:
import json
import requests
import folium
from datetime import date
from pathlib import Path
import pandas as pd
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import LineString



In [2]:
ROOT = Path("..").resolve()
EV_STATION_PATH = ROOT / "data/processed/ev_stations.geojson"
SUBSTATION_PATH = ROOT / "data/processed/substations.geojson"

ev_stations = gpd.read_file(EV_STATION_PATH)
substations = gpd.read_file(SUBSTATION_PATH)

In [3]:
stations_2263 = (
    ev_stations
    .to_crs(epsg=2263)
    [[
        "station_id",
        "station_name",
        "geometry"
    ]]
)

substations_2263 = (
    substations
    .to_crs(epsg=2263)
    .rename(columns={"name": "substation_name"})[[
        "substation_id",
        "substation_name",
        "geometry"
    ]]
)

mapping = stations_2263.sjoin_nearest(
    substations_2263,
    how="left",
    distance_col="distance_ft",
)


# Make LineString the default geometry for the mapping GeoDataFrame
# Use station geometry as the starting point for the LineString

mapping.geometry = mapping.apply(
    lambda row: LineString(
        [row["geometry"], substations_2263.loc[row["index_right"], "geometry"]]
    ),
    axis=1,
)


mapping["ev_station_count"] = (
    mapping.groupby("substation_id")
    ["station_id"].transform("nunique")
)

mapping

,station_id,station_name,geometry,index_right,substation_id,substation_name,distance_ft,ev_station_count
0,urn:agentwin:city:ev-charging-station:4e553805...,"ACS - 2554 Linden Blvd, Brooklyn","LINESTRING (1020327.926 182685.979, 1024460.31...",49,urn:agentwin:energy:substation:osm:383575265,151st Avenue Substation,4153.795722,5
1,urn:agentwin:city:ev-charging-station:78800793...,"ACS - 350 St Marks Pl, Staten Island","LINESTRING (962686.954 172713.141, 960247.845 ...",94,urn:agentwin:energy:substation:osm:383590989,Silver Lake Substation,3960.027989,8
2,urn:agentwin:city:ev-charging-station:5534de34...,"CITYHALL - New York City Hall, Manhattan","LINESTRING (982560.021 198971.008, 980850.383 ...",103,urn:agentwin:energy:substation:osm:1508559019,NaN,1727.153999,2
3,urn:agentwin:city:ev-charging-station:20506606...,"DCAS - 1 Centre St, Manhattan","LINESTRING (983090.915 199042.022, 984047.514 ...",22,urn:agentwin:energy:substation:osm:278084890,Peck Slip Substation,2069.804397,14
4,urn:agentwin:city:ev-charging-station:f97f5bcd...,"DCAS - 2 Navy St, Brooklyn","LINESTRING (989679.109 194528.187, 989280.725 ...",11,urn:agentwin:energy:substation:osm:250386730,Water Street Substation,1082.426856,16
...,...,...,...,...,...,...,...,...
513,urn:agentwin:city:ev-charging-station:3845beba...,"DCAS - 851 Grand Concourse, Bronx","LINESTRING (1005532.024 240238.123, 1001814.19...",19,urn:agentwin:energy:substation:osm:271609420,Parkview Substation,7635.602778,20
514,urn:agentwin:city:ev-charging-station:30de416d...,"DOE - 143 Baxter St, Manhattan","LINESTRING (984682.99 201192.276, 982455.709 2...",5,urn:agentwin:energy:substation:osm:219612704,Leonard Street Substation,2233.678408,4
515,urn:agentwin:city:ev-charging-station:4fafdcff...,"DSNY - 455 Park Avenue, Brooklyn","LINESTRING (995650.995 193043.062, 989280.725 ...",11,urn:agentwin:energy:substation:osm:250386730,Water Street Substation,6840.195300,16
516,urn:agentwin:city:ev-charging-station:3419ca6a...,"DPR - 151 97th Street Transverse, Manhattan","LINESTRING (995310.97 227229.994, 994311.065 2...",18,urn:agentwin:energy:substation:osm:271199175,West 110th Street Substation,4531.425108,7


In [4]:
# Plot the map (folium)

## Visually inspect which EV stations are closest to which substations and how many EV stations are mapped to each substation
## Goal: To ensure mapping is "balanced" (a roughly equal number of EV stations per substation)

map = mapping.explore(
    tiles="CartoDB positron",
)

folium.GeoJson(
    stations_2263.to_crs(epsg=4326),
    name="EV Stations",
    tooltip=folium.GeoJsonTooltip(fields=["station_name"]),
    popup=folium.GeoJsonPopup(fields=["station_id", "station_name"]),
    marker=folium.Marker(
        icon=folium.DivIcon(
            # icon_size=(12, 12),
            icon_anchor=(4, 4),            
            html="""
            <div style="
                width: 8px;
                height: 8px;
                background: #1769aa;
                border: 1px solid #0b3558;
                transform: rotate(45deg);
            "></div>
            """
        )
    ),
).add_to(map)

folium.GeoJson(
    substations_2263.to_crs(epsg=4326),
    name="Substations",
    tooltip=folium.GeoJsonTooltip(fields=["substation_name"]),
    popup=folium.GeoJsonPopup(fields=["substation_id", "substation_name"]),
    marker=folium.Circle(
        radius=57,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.9,
    )
).add_to(map)

map

In [5]:
# from branca.colormap import linear

# # Keep only the fields needed for the map and avoid ambiguous column names.
# substation_candidates = substations_2263[["substation_id", "name", "geometry"]].rename(
#     columns={"name": "substation_name"}
# )

# mapping = stations_2263.sjoin_nearest(
#     substation_candidates,
#     how="left",
#     distance_col="distance_ft",
# )

# # Folium requires longitude/latitude coordinates.
# mapping_wgs84 = mapping.to_crs(epsg=4326)
# substations_wgs84 = substations.to_crs(epsg=4326)

# # Lookup the assigned substation geometry for each station.
# substation_geometry = substations_wgs84.set_index("substation_id").geometry

# mapping_wgs84["assigned_substation_geometry"] = mapping_wgs84["substation_id"].map(
#     substation_geometry
# )

# # Assign each substation a stable color.
# substation_ids = mapping_wgs84["substation_id"].dropna().unique()
# colors = linear.Set1_09.scale(0, max(len(substation_ids) - 1, 1))

# substation_colors = {
#     substation_id: colors(index) for index, substation_id in enumerate(substation_ids)
# }

# # Center the map on the EV station network.
# map = folium.Map(
#     location=[
#         mapping_wgs84.geometry.y.mean(),
#         mapping_wgs84.geometry.x.mean(),
#     ],
#     zoom_start=10,
#     tiles="CartoDB positron",
# )

# # Add assignment lines first so markers remain visible.
# for _, row in mapping_wgs84.dropna(
#     subset=["geometry", "assigned_substation_geometry"]
# ).iterrows():
#     station_point = row.geometry
#     substation_point = row.assigned_substation_geometry

#     folium.PolyLine(
#         locations=[
#             [station_point.y, station_point.x],
#             [substation_point.y, substation_point.x],
#         ],
#         color=substation_colors.get(row["substation_id"], "#777777"),
#         weight=1,
#         opacity=0.35,
#     ).add_to(map)

# # EV station markers.
# ev_layer = folium.FeatureGroup(name="EV stations")

# for _, row in mapping_wgs84.iterrows():
#     distance_miles = row["distance_ft"] / 5280

#     popup = folium.Popup(
#         f"""
#         <b>{row["station_name"]}</b><br>
#         Borough: {row["borough"]}<br>
#         Ports: {row["no_of_ports"]}<br>
#         Assigned substation: {row["substation_name"]}<br>
#         Distance: {row["distance_ft"]:.0f} ft
#         ({distance_miles:.2f} mi)
#         """,
#         max_width=350,
#     )

#     folium.CircleMarker(
#         location=[row.geometry.y, row.geometry.x],
#         radius=5,
#         color="#1769aa",
#         fill=True,
#         fill_color="#2196f3",
#         fill_opacity=0.85,
#         popup=popup,
#     ).add_to(ev_layer)

# ev_layer.add_to(map)

# # Substation markers.
# substation_layer = folium.FeatureGroup(name="Con Edison substations")

# for _, row in substations_wgs84.iterrows():
#     color = substation_colors.get(row["substation_id"], "#b22222")

#     folium.CircleMarker(
#         location=[row.geometry.y, row.geometry.x],
#         radius=8,
#         color=color,
#         fill=True,
#         fill_color=color,
#         fill_opacity=1,
#         popup=folium.Popup(
#             f"""
#             <b>{row["name"]}</b><br>
#             Type: {row["substation"]}<br>
#             Voltage: {row["voltage"]}<br>
#             OSM ID: {row["id"]}
#             """,
#             max_width=300,
#         ),
#     ).add_to(substation_layer)

# substation_layer.add_to(map)

# folium.LayerControl().add_to(map)

# map